Ten skrypt konfiguruje agenta wyposażonego w narzędzia do przeszukiwania dokumentacji oraz internetu . Następnie skrypt uruchamia tego agenta w celu odpowiedzi na zapytania użytkownika, integrując go z platformą Arize Phoenix w celu śledzenia i ewaluacji jego działania, w tym poprawności wyboru narzędzi i trafności wyszukiwanych informacji.

# Setup

In [1]:
!pip install -qU "arize-phoenix>=8.0.0" datasets smolagents fsspec litellm phoenix openinference-instrumentation-smolagents langchain  rank_bm25 tqdm openai  langchain-community langchain-core duckduckgo_search

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.2/299.2 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.2/125.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.0/647.0 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.9/437.9 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 3.1 MB/s eta 0:00:00
   ━━━

**Lista instalowanych bibliotek i ich potencjalne zastosowania:**

*   `arize-phoenix`: Platforma do monitoringu i debugowania modeli uczenia maszynowego.
*   `datasets`: Biblioteka do łatwego dostępu i manipulacji zbiorami danych.
*   `smolagents`:  Biblioteka związana z agentami AI
*   `fsspec`: Interfejs do pracy z różnymi systemami plików, w tym chmurą.
*   `litellm`: Biblioteka ułatwiająca integrację z modelami językowymi (LLM).
*   `phoenix`:  monitorowanie i ewaluacja LLM  
*   `openinference-instrumentation-smolagents`: Narzędzia do monitorowania wydajności modeli podczas wnioskowania, w kontekście agentów AI.
*   `langchain`: Framework do tworzenia aplikacji opartych na modelach językowych.
*   `rank_bm25`: Algorytm rankingowy używany często w wyszukiwaniu informacji.
*   `tqdm`: Biblioteka do wyświetlania pasków postępu podczas długotrwałych operacji.
*   `openai`:  Biblioteka do interakcji z API OpenAI (np. GPT-3, GPT-4).
*   `langchain-community`: Zbiór modułów społecznościowych dla LangChain.
*   `langchain-core`: Podstawowe komponenty frameworka LangChain.
*   `duckduckgo_search`: Biblioteka do wykonywania wyszukiwań w DuckDuckGo.

In [ ]:
# Standard Library
import json
import os

# Third-Party Libraries - Data Handling & Machine Learning
import datasets

# Third-Party Libraries - Langchain
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever

# Third-Party Libraries - SmolAgents
from smolagents import CodeAgent, DuckDuckGoSearchTool, LiteLLMModel, Tool

# Third-Party Libraries - Phoenix & OpenInference (Observability & Tracing)
import phoenix as px
from phoenix.evals import OpenAIModel, llm_classify
from phoenix.trace import SpanEvaluations
from phoenix.trace.dsl import SpanQuery
from openinference.instrumentation import suppress_tracing
from openinference.instrumentation.smolagents import SmolagentsInstrumentor


from google.colab import userdata


Ten kod to sekcja importów bibliotek Pythona, które będą używane w programie. Można ją podzielić na kilka kategorii:

*   **Biblioteki standardowe:** `json` do pracy z danymi w formacie JSON oraz `os` do interakcji z systemem operacyjnym (np. dostęp do zmiennych środowiskowych, ścieżek plików).
*   **Biblioteki zewnętrzne – Obsługa danych i uczenie maszynowe:** `datasets` do pobierania i zarządzania zbiorami danych, oraz `pandas` do pracy z danymi tabelarycznymi w formacie DataFrame.
*   **Biblioteki zewnętrzne – Langchain:**  Importowane są klasy i funkcje z Langchain, takie jak `Document` (reprezentacja dokumentu tekstowego), `RecursiveCharacterTextSplitter` (do dzielenia tekstu na mniejsze fragmenty) oraz `BM25Retriever` (do wyszukiwania dokumentów przy użyciu algorytmu BM25).
*   **Biblioteki zewnętrzne – SmolAgents:** Importowane są kluczowe komponenty z biblioteki SmolAgents, w tym `CodeAgent` (agent wykonujący kod), `DuckDuckGoSearchTool` (narzędzie do wyszukiwania informacji w DuckDuckGo) oraz `LiteLLMModel` i `Tool` (klasy bazowe dla modeli językowych i narzędzi).
*   **Biblioteki zewnętrzne – Phoenix & OpenInference:** Importowane są biblioteki związane z monitorowaniem, śledzeniem i ewaluacją działania agentów AI.  Obejmuje to moduły z Phoenix (`px`, `phoenix.otel`, `phoenix.evals`, `phoenix.trace`) oraz narzędzia do integracji z OpenInference (`openinference.instrumentation`).

Kod ten przygotowuje środowisko programistyczne, importując wszystkie niezbędne biblioteki i klasy, które będą wykorzystywane w dalszej części programu do budowy, uruchamiania i monitorowania agentów AI.

In [10]:
class CFG:
  model = 'gpt-4o-mini'


os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "https://app.phoenix.arize.com"
os.environ["PHOENIX_CLIENT_HEADERS"] = "api_key=" + userdata.get('arize')
os.environ["OPENAI_API_KEY"] = userdata.get('openaivision')

openai_key = userdata.get('openaivision')

# Narzędzia

## Pozyskiwacz

In [ ]:
knowledge_base = datasets.load_dataset("m-ric/huggingface_doc", split="train")

knowledge_base = knowledge_base.filter(lambda row: row["source"].startswith("huggingface/transformers"))

README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

huggingface_doc.csv:   0%|          | 0.00/22.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2647 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2647 [00:00<?, ? examples/s]

In [5]:
source_docs = [
    Document(page_content=doc["text"], metadata={"source": doc["source"].split("/")[1]})
    for doc in knowledge_base
]

# Initialize Text Splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    add_start_index=True,
    strip_whitespace=True,
    separators=["\n\n", "\n", ". ", " ", ""], # Adjusted separators slightly
)
docs_processed = text_splitter.split_documents(source_docs)
print(f"Processed {len(source_docs)} documents into {len(docs_processed)} chunks.")



Processed 484 documents into 14755 chunks.


In [6]:
class RetrieverTool(Tool):
    name = "retriever" # How the agent refers to the tool
    description = ( # Description helps the agent decide WHEN to use the tool
        "Uses semantic search (BM25) to retrieve the parts of "
        "Transformers documentation that could be most relevant to answer your query."
    )
    inputs = { # Defines expected input arguments for the LLM
        "query": {
            "type": "string",
            "description": (
                "The query to perform. This should be semantically close to your target "
                "documents. Use the affirmative form rather than a question for better results."
            ),
        }
    }
    output_type = "string" # Defines the expected output type

    def __init__(self, docs, **kwargs):
        super().__init__(**kwargs)
        print("Initializing BM25 Retriever...")
        # Initialize the retriever with the processed documents
        self.retriever = BM25Retriever.from_documents(docs, k=10) # Retrieve top 10 chunks
        print("Retriever initialized.")

    def forward(self, query: str) -> str:
        """The actual method called when the agent uses the tool."""
        print(f"RetrieverTool received query: {query}")
        assert isinstance(query, str), "Your search query must be a string"

        # Perform retrieval
        docs = self.retriever.invoke(query)

        # Format the output for the agent
        output = "\nRetrieved documents:\n" + "".join(
            [
                f"\n\n===== Document {str(i)} =====\n" + doc.page_content
                for i, doc in enumerate(docs)
            ]
        )
        # Optional: print snippet of output for debugging
        # print(f"RetrieverTool returning {len(docs)} documents (first 100 chars): {output[:100]}...")
        return output


Podsumowując, klasa `RetrieverTool` implementuje narzędzie do semantycznego wyszukiwania w bazie wiedzy przy użyciu algorytmu BM25. Narzędzie to jest przeznaczone do integracji z agentem AI i pomaga mu znaleźć odpowiednie informacje w dokumentach.

In [7]:
retriever_tool = RetrieverTool(docs=docs_processed)


Initializing BM25 Retriever...
Retriever initialized.


Ten kod tworzy instancję klasy `RetrieverTool`, przekazując jej przetworzone fragmenty tekstu (`docs_processed`) jako argument.

*   `retriever_tool = RetrieverTool(docs=docs_processed)`:  Tworzy nowy obiekt o nazwie `retriever_tool` z klasy `RetrieverTool`.
*   `docs=docs_processed`: Przekazuje listę fragmentów tekstu (`docs_processed`), które zostały wcześniej wygenerowane przez podział oryginalnej bazy wiedzy, do konstruktora klasy `RetrieverTool`.  Te fragmenty będą używane przez retriever BM25 do wyszukiwania informacji.

W efekcie tego kodu tworzony jest obiekt narzędzia do wyszukiwania, który jest gotowy do użycia przez agenta AI. Obiekt ten zawiera zainicjalizowany retriever BM25 z załadowanymi i przetworzonymi dokumentami.

## Wyszukiwarka

In [8]:
search_tool = DuckDuckGoSearchTool()

# Model

In [11]:

model = LiteLLMModel(
    model_id = CFG.model,
    api_key=openai_key
)
print(f"Using LLM: {model.model_id}")


Using LLM: gpt-4o-mini


# Agent

Na razie nie śledzony.

In [12]:
# Create the agent instance
manager_agent = CodeAgent(
    tools=[retriever_tool, search_tool], # Pass instances of our tools
    model=model
)

In [13]:


# Initial Test Run (Before Tracing)
print("\n--- Running Agent (Initial Untraced Run) ---")
user_query = "For a transformers model training, which is slower, the forward or the backward pass?"
print(f"User Query: {user_query}")


final_answer = manager_agent.run(user_query)

print("\n--- Agent Finished ---")
print(f"Final Answer: {final_answer}")


--- Running Agent (Initial Untraced Run) ---
User Query: For a transformers model training, which is slower, the forward or the backward pass?


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ For a transformers model training, which is slower, the forward or the backward pass?                           │
│                                                                                                                 │
╰─ LiteLLMModel - gpt-4o-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  information = retriever(query="transformers model training forward vs backward pass speed")                      
  print(information)                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

RetrieverTool received query: transformers model training forward vs backward pass speed


...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("The backward pass is slower than the forward pass during transformers model training.")            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: The backward pass is slower than the forward pass during transformers model training.

[Step 2: Duration 2.51 seconds| Input tokens: 5,065 | Output tokens: 192]


--- Agent Finished ---
Final Answer: The backward pass is slower than the forward pass during transformers model training.


# Test

In [14]:


PROJECT_NAME = "smolagent_evaluation_demo"

print("Initializing Phoenix OpenTelemetry integration...")
tracer_provider = phoenix.otel.register(
    project_name=PROJECT_NAME,
    endpoint="https://app.phoenix.arize.com/v1/traces",
)
print(f"Traces will be sent to Phoenix project: {PROJECT_NAME}")

Initializing Phoenix OpenTelemetry integration...
🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: smolagent_evaluation_demo
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: https://app.phoenix.arize.com/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {'api_key': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

Traces will be sent to Phoenix project: smolagent_evaluation_demo


Ten kod inicjalizuje integrację z systemem śledzenia OpenTelemetry za pomocą biblioteki Phoenix, umożliwiając monitorowanie i debugowanie działania aplikacji.

*   `PROJECT_NAME = "smolagent_evaluation_demo"`: Definiuje nazwę projektu jako ciąg znaków. Ta nazwa będzie używana do identyfikacji danych śledzenia w systemie Phoenix.
*   `print("Initializing Phoenix OpenTelemetry integration...")`: Wyświetla komunikat informujący o inicjalizacji integracji z OpenTelemetry.
*   `tracer_provider = phoenix.otel.register(...)`: Rejestruje dostawcę śledzenia (tracer provider) za pomocą funkcji `phoenix.otel.register()`. Dostawca śledzenia jest odpowiedzialny za przechwytywanie i eksportowanie danych śledzenia.
    *   `project_name=PROJECT_NAME`: Ustawia nazwę projektu na wartość zmiennej `PROJECT_NAME`.
    *   `endpoint="http://127.0.0.1:6006/v1/traces"`: Określa adres URL punktu końcowego, do którego będą wysyłane dane śledzenia. W tym przypadku jest to lokalny serwer Phoenix działający na porcie 6006.
*   `print(f"Traces will be sent to Phoenix project: {PROJECT_NAME}")`: Wyświetla komunikat informujący o tym, że dane śledzenia będą wysyłane do projektu o nazwie `PROJECT_NAME`.

Podsumowując, kod ten konfiguruje system śledzenia OpenTelemetry, który pozwoli na monitorowanie i debugowanie działania aplikacji. Dane śledzenia będą wysyłane do lokalnego serwera Phoenix, gdzie można je analizować i wizualizować.

In [15]:

print("Instrumenting Smolagents...")
SmolagentsInstrumentor().instrument()
print("Smolagents instrumentation complete.")

Instrumenting Smolagents...
Smolagents instrumentation complete.


Ten kod instrumentuje bibliotekę Smolagents w celu zbierania danych śledzenia za pomocą OpenTelemetry, które wcześniej zostały skonfigurowane.

*   `print("Instrumenting Smolagents...")`: Wyświetla komunikat informujący o rozpoczęciu procesu instrumentacji Smolagents.
*   `SmolagentsInstrumentor().instrument()`: Tworzy instancję klasy `SmolagentsInstrumentor` i wywołuje jej metodę `instrument()`. Ta metoda dodaje kod śledzenia do biblioteki Smolagents, co pozwala na przechwytywanie informacji o działaniu tej biblioteki.
*   `print("Smolagents instrumentation complete.")`: Wyświetla komunikat informujący o pomyślnym zakończeniu instrumentacji Smolagents.

Podsumowując, kod ten przygotowuje bibliotekę Smolagents do śledzenia jej działania za pomocą OpenTelemetry. Dzięki temu można monitorować i analizować działanie agentów AI opartych na Smolagents w systemie Phoenix.

In [31]:
# user_query = "For a transformers model training, which is slower, the forward or the backward pass?"
user_query = "What is the most recent version of the transformers library?"
print(f"User Query: {user_query}")

# Execute the agent run (tracing)
final_answer = manager_agent.run(user_query)

print("\n--- Agent Finished (Traced) ---")
print(f"Final Answer: {final_answer}")
print("\nTrace data should now be available in Phoenix.")

User Query: What is the most recent version of the transformers library?


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the most recent version of the transformers library?                                                    │
│                                                                                                                 │
╰─ LiteLLMModel - gpt-4o-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  latest_version_info = web_search("latest version of transformers library")                                       
  print(latest_version_info)                                                                                       
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results

[transformers · PyPI](https://pypi.org/project/transformers/)
Transformers is a library of pretrained text, computer vision, audio, video, and multimodal models for inference 
and training. Use Transformers to fine-tune models on your data, build inference applications, and for generative 
AI use cases across multiple modalities. ... However, the latest version may not be stable. Feel free to open an 
issue ...

[Releases · huggingface/transformers - GitHub](https://github.com/huggingface/transformers/releases)
A new model is added to transformers: Qwen2.5-Omni. It is added on top of the v4.51.3 release, and can be installed
from the following tag: ... In this work, we introduce Janus-Pro, an advanced version of the previous work Janus. 
Specifically, Janus-Pro incorporates (1) an optimized training strate (2) expanded training data, and (3) scaling 
to ...

[Installation — transformers 4.7.0 documentation - Hugging 
Face](https://huggingface.co/transformers/v4.7.0/installation.html)
State-of-the-art Natural Language Processing for PyTorch and TensorFlow 2.0. Transformers provides thousands of 
pretrained models to perform tasks on texts such as classification, information extraction, question answering, 
summarization, translation, text generation, etc in 100+ languages. Its aim is to make cutting-edge NLP easier to 
use for everyone

[Transformers - Anaconda.org](https://anaconda.org/conda-forge/transformers)
State-of-the-art Natural Language Processing for TensorFlow 2.0 and PyTorch. copied from cf-post-staging / 
transformers

[Transformers - Hugging Face](https://huggingface.co/docs/transformers/v4.17.0/en/index)
🤗 Transformers State-of-the-art Machine Learning for PyTorch, TensorFlow and JAX. 🤗 Transformers provides APIs to
easily download and train state-of-the-art pretrained models. Using pretrained models can reduce your compute 
costs, carbon footprint, and save you time from training a model from scratch.

[transformers published releases on PyPI - Libraries.io - security 
...](https://libraries.io/pypi/transformers/versions)
transformers Releases 4.48.2: January 30th, 2025 19:52 Browse source on GitHub View diff between 4.48.2 and 4.48.1 
4.48.1: January 20th, 2025 16:36 Browse source on GitHub View diff between 4.48.1 and 4.48.0 4.48.0: January 10th, 
2025 12:14 Browse source on GitHub ...

[GitHub - huggingface/transformers: Transformers: State-of-the-art 
...](https://github.com/huggingface/transformers)
Transformers is a library of pretrained text, computer vision, audio, video, and multimodal models for inference 
and training. Use Transformers to fine-tune models on your data, build inference applications, and for generative 
AI use cases across multiple modalities. ... However, the latest version may not be stable. Feel free to open an 
issue ...

[How often are new versions of transformers library 
released?](https://discuss.huggingface.co/t/how-often-are-new-versions-of-transformers-library-released/26363)
Hey, When is the next version of transformers library going to be released? There are some crucial pull requests 
merged, which I'd like to access. So now I'm pondering whether to construct some temporary solution with my own 
docker image, or wait for the release 🙂

[Upgrading - Simple Transformers](https://simpletransformers.ai/docs/upgrading/)
Simple Transformers is updated regularly and using the latest version is highly recommended. This will ensure that 
you have access to the latest features, improvements, and bug fixes. ... version with pip, you can do; 1 pip show 
simpletransformers As Simple Transformers is built on top of the Hugging Face Transformers library, make sure that 
...

[Installation - Hugging Face](https://huggingface.co/docs/transformers/en/installation)
Source install. Installing from source installs the latest version rather than the stable version of the library. 
It ensures you have the most up-to-date changes in Transformers and it's useful 

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # From the previous observation, it seems the highest version mentioned was 4.51.3                               
  latest_version = "4.51.3"                                                                                        
  final_answer(latest_version)                                                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: 4.51.3

[Step 3: Duration 1.70 seconds| Input tokens: 8,596 | Output tokens: 268]


--- Agent Finished (Traced) ---
Final Answer: 4.51.3

Trace data should now be available in Phoenix.


Ten kod ponownie uruchamia agenta AI z tym samym zapytaniem użytkownika, ale tym razem z włączonym śledzeniem OpenTelemetry.

*   `user_query = "For a transformers model training, which is slower, the forward or the backward pass?"`: Definiuje zapytanie użytkownika (takie samo jak poprzednio).
*   `print(f"User Query: {user_query}")`: Wyświetla zapytanie użytkownika.
*   `final_answer = manager_agent.run(user_query)`: Uruchamia agenta AI (`manager_agent`) i przekazuje mu zapytanie użytkownika. Ponieważ wcześniej zainicjowano integrację z OpenTelemetry i instrumentowano Smolagents, wszystkie operacje wykonywane przez agenta będą teraz śledzone.
*   `print("\n--- Agent Finished (Traced) ---")`: Wyświetla komunikat informujący o zakończeniu działania agenta w trybie ze śledzeniem.
*   `print(f"Final Answer: {final_answer}")`: Wyświetla odpowiedź wygenerowaną przez agenta AI.
*   `print("\nTrace data should now be available in Phoenix.")`: Wyświetla komunikat informujący, że dane śledzenia powinny być teraz dostępne w systemie Phoenix. Można je przeanalizować i wizualizować w interfejsie użytkownika Phoenix, aby zrozumieć, jak agent działał podczas odpowiadania na zapytanie.

Podsumowując, kod ten uruchamia agenta AI z włączonym śledzeniem, co pozwala na monitorowanie jego działania i debugowanie ewentualnych problemów. Dane śledzenia są wysyłane do systemu Phoenix, gdzie można je analizować.

# Test 2: narzędzia

In [32]:

print("Querying LLM spans for tool choice evaluation...")
llm_query = SpanQuery().where(
    "span_kind == 'LLM'" # Target the LLM reasoning spans
).select(
    span_id="context.span_id", # Get span_id for logging
    question="input.value",       # The input prompt to the LLM
    generated_code="output.value" # Get the LLM's generated code output
)

llm_decision_df = px.Client().query_spans(llm_query,
                                           project_name=PROJECT_NAME,
                                           timeout=None)

Querying LLM spans for tool choice evaluation...


/usr/local/lib/python3.11/dist-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (8.27.0) and client (9.6.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(


Ten kod pobiera dane o wywołaniach modelu językowego (LLM) z Arize Phoenix w celu oceny procesu wyboru narzędzi przez agenta.

*   `print("Querying LLM spans for tool choice evaluation...")`: Wyświetla komunikat informujący o rozpoczęciu zapytania do Arize Phoenix w celu pobrania danych o wywołaniach LLM.
*   `llm_query = SpanQuery().where(...)`: Tworzy obiekt `SpanQuery`, który definiuje kryteria wyszukiwania danych śledzenia (spans) w Arize Phoenix.
    *   `.where("span_kind == 'LLM'")`: Określa warunek, że mają być wybrane tylko spany, których typ (`span_kind`) to "LLM". Oznacza to, że zapytanie dotyczy tylko tych fragmentów śledzenia, które reprezentują wywołania modelu językowego.
*   `.select(...)`: Definiuje, jakie dane mają być pobrane z wybranych spanów.
    *   `span_id="context.span_id"`: Pobiera identyfikator spanu (`span_id`) z kontekstu spanu. Identyfikator ten jest unikalnym identyfikatorem danego wywołania LLM.
    *   `question="input.value"`: Pobiera zapytanie (prompt) wysłane do modelu językowego (`input.value`).
    *   `generated_code="output.value"`: Pobiera kod wygenerowany przez model językowy (`output.value`).
*   `llm_decision_df = px.Client().query_spans(...)`: Wykonuje zapytanie do Arize Phoenix i pobiera dane śledzenia.
    *   `px.Client().query_spans(llm_query, ...)`: Używa klienta Arize Phoenix (`px.Client()`) do wysłania zapytania `llm_query`.
    *   `project_name=PROJECT_NAME`: Określa nazwę projektu w Arize Phoenix, z którego mają być pobierane dane.
    *   `timeout=None`: Ustawia limit czasu na wykonanie zapytania na nieskończoność (brak limitu).

Podsumowując, kod ten wysyła zapytanie do Arize Phoenix w celu pobrania danych o wywołaniach modelu językowego, w tym zapytań i wygenerowanego kodu. Dane te są przechowywane w ramce danych (`llm_decision_df`) i mogą być wykorzystane do analizy procesu podejmowania decyzji przez agenta AI, a konkretnie tego, jak wybiera narzędzia na podstawie odpowiedzi modelu językowego.

In [33]:
# Filter for spans that actually generated code calling our tools
llm_decision_df = llm_decision_df[
    llm_decision_df['generated_code'].astype(str).str.contains("retriever|web_search", na=False)
].copy()

print(f"Found {len(llm_decision_df)} relevant LLM decision spans.")

Found 0 relevant LLM decision spans.


Ten kod filtruje ramkę danych `llm_decision_df`, zachowując tylko te wiersze, które reprezentują wywołania modelu językowego, w których model wygenerował kod odwołujący się do narzędzi "retriever" lub "web_search".

*   `llm_decision_df = llm_decision_df[...]`: Przypisuje przefiltrowaną ramkę danych z powrotem do zmiennej `llm_decision_df`.
*   `llm_decision_df['generated_code'].astype(str).str.contains("retriever|web_search", na=False)`: To wyrażenie filtrujące, które sprawdza, czy kolumna 'generated_code' zawiera ciągi znaków "retriever" lub "web_search".
    *   `llm_decision_df['generated_code']`: Wybiera kolumnę 'generated_code' z ramki danych.
    *   `.astype(str)`: Konwertuje wartości w kolumnie na typ string (ciąg znaków). Jest to ważne, aby uniknąć błędów, jeśli kolumna zawiera inne typy danych.
    *   `.str.contains("retriever|web_search", na=False)`: Sprawdza, czy każdy ciąg znaków w kolumnie zawiera podciąg "retriever" lub "web_search". Operator `|` oznacza "lub".  `na=False` oznacza, że wartości brakujące (NaN) są traktowane jako False – czyli wiersze z brakującymi danymi w kolumnie 'generated_code' nie zostaną uwzględnione.
*   `.copy()`: Tworzy kopię przefiltrowanej ramki danych. Jest to ważne, aby uniknąć ostrzeżeń związanych ze zmianą kopii ramki danych.

*   `print(f"Found {len(llm_decision_df)} relevant LLM decision spans.")`: Wyświetla komunikat informujący o liczbie wierszy (spanów) w przefiltrowanej ramce danych.  Wskazuje to, ile razy model językowy wygenerował kod odwołujący się do narzędzi "retriever" lub "web_search".

Podsumowując, kod ten filtruje dane śledzenia, aby skupić się tylko na tych przypadkach, w których model językowy podjął decyzję o użyciu konkretnych narzędzi (retriever i web search). Pozwala to na analizę tego, jakie czynniki wpływają na wybór narzędzi przez agenta AI.

In [19]:
EVAL_GENERATED_CODE_TEMPLATE = """
You are an evaluation assistant. Your task is to determine if the generated Python code correctly uses an available tool to address the user's question.

Available Tools:
- retriever: Uses semantic search on Transformers documentation. Use for documentation-specific questions.
- duckduckgo_search: Searches the web. Use for general knowledge or non-documentation questions.

[BEGIN DATA]
************
[Question]: {question}
************
[Generated Code]: {generated_code}
[END DATA]

Based on the Question, does the Generated Code call the *most appropriate* tool (retriever or duckduckgo_search) with a *reasonable* query argument?

Your response must be a single word, either "correct" or "incorrect".
- "correct" means the most appropriate tool was chosen and the argument seems relevant to the question.
- "incorrect" means the wrong tool was chosen, the argument is unrelated, or the code is malformed/doesn't call a tool.
"""

In [30]:
if not llm_decision_df.empty:
    print("Running tool choice evaluation...")
    with suppress_tracing():
        tool_choice_eval = llm_classify(
            dataframe=llm_decision_df,
            template=EVAL_GENERATED_CODE_TEMPLATE,
            model=OpenAIModel(model="gpt-4o"), # Choose your judge model
            rails=['correct', 'incorrect'],
            provide_explanation=True
        )
    tool_choice_eval['score'] = tool_choice_eval.apply(lambda x: 1 if x['label']=='correct' else 0, axis=1)
    print("Tool choice evaluation complete.")

    # Attach scores back to the original LLM spans in Phoenix
    print("Logging tool choice evaluations...")
    px.Client().log_evaluations(
        SpanEvaluations(eval_name="Tool Choice Eval", dataframe=tool_choice_eval)
    )
    print("Tool choice evaluations logged.")
else:
    print("No relevant LLM decision spans found for tool choice evaluation.")

No relevant LLM decision spans found for tool choice evaluation.


Ten kod przeprowadza ocenę wyboru narzędzi przez agenta AI, wykorzystując model językowy do klasyfikacji wygenerowanego kodu. Wyniki oceny są następnie logowane w Arize Phoenix.

*   `if not llm_decision_df.empty:`: Sprawdza, czy ramka danych `llm_decision_df` nie jest pusta. Oznacza to, że znaleziono co najmniej jeden przypadek wywołania modelu językowego z kodem odwołującym się do narzędzi.
*   `print("Running tool choice evaluation...")`: Wyświetla komunikat informujący o rozpoczęciu oceny wyboru narzędzi.
*   `with suppress_tracing():`:  Wyłącza śledzenie OpenTelemetry podczas wykonywania tej części kodu. Jest to przydatne, ponieważ ocena wyboru narzędzi jest sama w sobie procesem, który nie wymaga szczegółowego śledzenia.
    *   `tool_choice_eval = llm_classify(...)`: Wywołuje funkcję `llm_classify`, która przeprowadza klasyfikację danych z ramki danych `llm_decision_df` przy użyciu modelu językowego.
        *   `dataframe=llm_decision_df`: Przekazuje ramkę danych zawierającą dane o wywołaniach LLM.
        *   `template=EVAL_GENERATED_CODE_TEMPLATE`: Przekazuje szablon zapytania, który będzie używany do generowania zapytań dla modelu językowego.
        *   `model=OpenAIModel(model="gpt-4o")`: Określa model językowy, który ma być użyty do klasyfikacji – w tym przypadku "gpt-4o" z OpenAI.
        *   `rails=['correct', 'incorrect']`: Definiuje możliwe etykiety (klasy) dla oceny – "correct" i "incorrect".
        *   `provide_explanation=True`: Włącza generowanie wyjaśnień przez model językowy, dlaczego podjął daną decyzję.
    *   `tool_choice_eval['score'] = tool_choice_eval.apply(...)`: Dodaje nową kolumnę o nazwie "score" do ramki danych `tool_choice_eval`. Wartość w tej kolumnie jest ustawiana na 1, jeśli etykieta (kolumna 'label') to "correct", a na 0 w przeciwnym razie.
*   `print("Tool choice evaluation complete.")`: Wyświetla komunikat informujący o zakończeniu oceny wyboru narzędzi.
*   `print("Logging tool choice evaluations...")`: Wyświetla komunikat informujący o rozpoczęciu logowania wyników oceny do Arize Phoenix.
    *   `px.Client().log_evaluations(...)`: Loguje wyniki oceny do Arize Phoenix.
        *   `SpanEvaluations(eval_name="Tool Choice Eval", dataframe=tool_choice_eval)`: Tworzy obiekt `SpanEvaluations`, który zawiera nazwę oceny ("Tool Choice Eval") i ramkę danych z wynikami (`tool_choice_eval`).
    *   `print("Tool choice evaluations logged.")`: Wyświetla komunikat informujący o pomyślnym zalogowaniu wyników oceny.
*   `else:`: Jeśli ramka danych `llm_decision_df` jest pusta (nie znaleziono żadnych odpowiednich wywołań LLM).
    *   `print("No relevant LLM decision spans found for tool choice evaluation.")`: Wyświetla komunikat informujący, że nie znaleziono odpowiednich spanów do oceny.

Podsumowując, kod ten przeprowadza automatyczną ocenę jakości wyboru narzędzi przez agenta AI, wykorzystując model językowy jako sędziego. Wyniki oceny są logowane w Arize Phoenix, co pozwala na monitorowanie i analizę wydajności agenta.

# Test 3: istotność

In [21]:
retriever_query = SpanQuery().where(
    "span_kind == 'TOOL' and tool.name == 'retriever'"
).select(
    trace_id="context.trace_id",
    span_id="context.span_id",
    retriever_input="input.value",
    retrieved_docs="output.value",

)

retriever_spans_df = px.Client().query_spans(retriever_query,
                                             project_name=PROJECT_NAME,
                                             timeout=None)

if retriever_spans_df.empty:
    print("Query still returned empty. Double-check project_name and trace data existence.")

retriever_spans_df

/usr/local/lib/python3.11/dist-packages/phoenix/utilities/client.py:51: UserWarning: ⚠️⚠️ The Phoenix server (8.27.0) and client (9.6.0) versions are severely mismatched. Upgrade  either the client or server to ensure API compatibility ⚠️⚠️
  warnings.warn(


,context.trace_id,retriever_input,retrieved_docs
context.span_id,,,
9dac2dd1bd6d3d30,18b319abc5aafb3dd9f6a30c7e706308,"{""args"": [], ""sanitize_inputs_outputs"": false,...",\nRetrieved documents:\n\n\n===== Document 0 =...


Ten kod pobiera dane o wywołaniach narzędzia "retriever" z Arize Phoenix, w celu dalszej analizy i oceny jego działania.

*   `retriever_query = SpanQuery().where(...)`: Tworzy obiekt `SpanQuery`, który definiuje kryteria wyszukiwania danych śledzenia (spans) w Arize Phoenix.
    *   `.where("span_kind == 'TOOL' and tool.name == 'retriever'")`: Określa warunek, że mają być wybrane tylko spany, których typ (`span_kind`) to "TOOL" i nazwa narzędzia (`tool.name`) to "retriever". Oznacza to, że zapytanie dotyczy tylko tych fragmentów śledzenia, które reprezentują wywołania narzędzia retriever.
*   `.select(...)`: Definiuje, jakie dane mają być pobrane z wybranych spanów.
    *   `trace_id="context.trace_id"`: Pobiera identyfikator śladu (`trace_id`) z kontekstu spanu. Identyfikator śladu grupuje wszystkie spany związane z jednym żądaniem lub zadaniem.
    *   `span_id="context.span_id"`: Pobiera identyfikator spanu (`span_id`) z kontekstu spanu.
    *   `retriever_input="input.value"`: Pobiera zapytanie (prompt) wysłane do narzędzia retriever (`input.value`).
    *   `retrieved_docs="output.value"`: Pobiera dokumenty zwrócone przez narzędzie retriever (`output.value`).
*   `retriever_spans_df = px.Client().query_spans(...)`: Wykonuje zapytanie do Arize Phoenix i pobiera dane śledzenia.
    *   `px.Client().query_spans(retriever_query, ...)`: Używa klienta Arize Phoenix (`px.Client()`) do wysłania zapytania `retriever_query`.
    *   `project_name=PROJECT_NAME`: Określa nazwę projektu w Arize Phoenix, z którego mają być pobierane dane.
    *   `timeout=None`: Ustawia limit czasu na wykonanie zapytania na nieskończoność (brak limitu).
*   `if retriever_spans_df.empty:`: Sprawdza, czy ramka danych `retriever_spans_df` jest pusta. Oznacza to, że nie znaleziono żadnych wywołań narzędzia "retriever" spełniających kryteria zapytania.
    *   `print("Query still returned empty. Double-check project_name and trace data existence.")`: Wyświetla komunikat informujący o tym, że zapytanie zwróciło pusty wynik i sugeruje sprawdzenie nazwy projektu oraz istnienia danych śledzenia.
*   `retriever_spans_df`:  Jeśli ramka danych nie jest pusta, zwraca ją jako wynik działania tego bloku kodu.

Podsumowując, kod ten pobiera dane o wywołaniach narzędzia retriever z Arize Phoenix, w tym zapytania i zwrócone dokumenty. Dane te mogą być wykorzystane do analizy skuteczności narzędzia retriever oraz oceny jakości wyników wyszukiwania.

In [22]:
EVAL_RETRIEVAL_RELEVANCE_TEMPLATE = """
You are an evaluation assistant. Your task is to assess the relevance of retrieved documents for a given query.

[BEGIN DATA]
************
[Query Sent to Retriever]: {retriever_query}
************
[Retrieved Documents]:
{retrieved_docs}
************
[END DATA]

Based *only* on the [Query Sent to Retriever] and the content of the [Retrieved Documents], are the documents relevant to the query?

- "Relevant": The documents contain information directly related to the topics or entities mentioned in the query.
- "Irrelevant": The documents do not contain information related to the topics or entities mentioned in the query.

Your response must be a single word, either "Relevant" or "Irrelevant".
"""

Ten kod definiuje szablon zapytania dla modelu językowego, który będzie używany do oceny trafności dokumentów zwróconych przez narzędzie retriever w odpowiedzi na dane zapytanie.

*   `EVAL_RETRIEVAL_RELEVANCE_TEMPLATE = """..."""`: Definiuje zmienną `EVAL_RETRIEVAL_RELEVANCE_TEMPLATE`, która przechowuje wielowierszowy ciąg znaków reprezentujący szablon zapytania.
*   Szablon ten zawiera instrukcje dla modelu językowego, opisujące jego rolę (jako asystenta oceniającego) i zadanie (ocena trafności dokumentów).
*   `[BEGIN DATA] ... [END DATA]`: Oznacza sekcję danych wejściowych dla modelu językowego.
    *   `[Query Sent to Retriever]: {retriever_query}`: Miejsce zastępcze dla zapytania wysłanego do narzędzia retriever.
    *   `[Retrieved Documents]:\n{retrieved_docs}`: Miejsce zastępcze dla dokumentów zwróconych przez narzędzie retriever.  Dodanie `\n` powoduje wstawienie nowego wiersza przed treścią dokumentów, co poprawia czytelność.
*   `Based *only* on the [Query Sent to Retriever] and the content of the [Retrieved Documents], are the documents relevant to the query?`: Pytanie kierowane do modelu językowego, które prosi o ocenę trafności dokumentów w odniesieniu do zapytania. Podkreśla się, że ocena ma być oparta *wyłącznie* na zawartości zapytania i dokumentów.
*   `Your response must be a single word, either "Relevant" or "Irrelevant".`: Określa format odpowiedzi – model ma zwrócić tylko jedno słowo: "Relevant" (trafne) lub "Irrelevant" (nietrafne). Definiuje również kryteria oceny dla każdego z tych słów.

Podsumowując, ten kod definiuje szablon zapytania, który będzie używany do automatycznej oceny trafności dokumentów zwróconych przez narzędzie retriever. Szablon ten zawiera instrukcje i dane wejściowe potrzebne modelowi językowemu do podjęcia decyzji o trafności dokumentów.

In [25]:
def extract_query(input_val):
    try:
        input_dict = json.loads(input_val)
        if isinstance(input_dict, dict) and 'args' in input_dict and len(input_dict['args']) > 0:
            return input_dict['args'][0]
    except (json.JSONDecodeError, TypeError, KeyError, IndexError):
            if isinstance(input_val, str) and not input_val.strip().startswith('{'):
                return input_val
    return None

if not retriever_spans_df.empty:
    # Create the 'retriever_query' column (index is preserved)
    retriever_spans_df['retriever_query'] = retriever_spans_df['retriever_input'].apply(extract_query)

    # Select only the columns needed for the template, the index ('context.span_id') is kept automatically
    retriever_eval_df = retriever_spans_df[['retriever_query', 'retrieved_docs']].copy()

    # Drop rows where parsing failed or docs are missing
    retriever_eval_df = retriever_eval_df.dropna(subset=['retriever_query', 'retrieved_docs'])

In [28]:
if not retriever_eval_df.empty:
    print("Data prepared for retrieval evaluation (index is context.span_id):")
    print(retriever_eval_df.head())

    print("\nRunning retrieval relevance evaluation...")
    with suppress_tracing():
        retrieval_relevance_eval = llm_classify(
            dataframe=retriever_eval_df,
            template=EVAL_RETRIEVAL_RELEVANCE_TEMPLATE,
            rails=['relevant', 'irrelevant'],
            model=OpenAIModel(model="gpt-4o"),
            provide_explanation=True
        )

    retrieval_relevance_eval['score'] = retrieval_relevance_eval.apply(lambda x: 1 if x['label']=='relevant' else 0, axis=1)

    print("\nRetrieval Relevance Evaluation Results:")
    print(retrieval_relevance_eval.head()) # This DataFrame will also have 'context.span_id' as index

    px.Client().log_evaluations(
          SpanEvaluations(eval_name="Retrieval Relevance Eval", dataframe=retrieval_relevance_eval)
      )